# Data audit / EDA

Roadmap stage 3. This notebook is self-contained: it downloads the
dataset itself (same as `01_dataset_download_colab.ipynb`), so it can be
run on its own in a fresh Colab session without depending on another
notebook having run first.

## Step 3.1 — Verify dataset integrity

Goal: confirm all 200 class folders are present, and that every image file
actually opens correctly (not just a sample — with ~116K files this is
fast enough to check all of them and get a definitive answer).

### Setup: download the dataset (from Google Drive)

In [ ]:
!pip install -q gdown

DRIVE_URL = "https://drive.google.com/file/d/1Mi0IleRucNmwnQ4g_ZEWOyBFFv4mO9ba/view?usp=sharing"
ZIP_PATH = "/content/traffic_sign_dataset.zip"

!gdown --fuzzy "$DRIVE_URL" -O "$ZIP_PATH"

In [ ]:
!mkdir -p /content/dataset
!unzip -q "$ZIP_PATH" -d /content/dataset
!rm "$ZIP_PATH"

from pathlib import Path
data_path = Path('/content/dataset/Data')
assert data_path.is_dir(), (
    "Unzip did not produce /content/dataset/Data as expected — the zip may be "
    "corrupt or incomplete. Re-run the download cell and check its output for errors."
)
n_classes = len(list(data_path.iterdir()))
print(f"Unzip done, zip file removed. Found {n_classes} folders under Data/.")

In [ ]:
!mkdir -p /content/dataset
!unzip -q "$ZIP_PATH" -d /content/dataset
!rm "$ZIP_PATH"
print("Unzip done, zip file removed to free up session disk.")

### 3.1a — Confirm all 200 class folders are present

In [ ]:
from pathlib import Path

data_dir = Path('/content/dataset/Data')
class_folders = sorted(data_dir.iterdir(), key=lambda p: int(p.name))

found = set(p.name for p in class_folders)
expected = set(str(i) for i in range(200))

print(f"Found {len(class_folders)} class folders")
print("Missing classes:", sorted(expected - found, key=int) or "None")
print("Unexpected extra folders:", sorted(found - expected) or "None")

### 3.1b — Verify every image actually opens (full check, not a sample)

In [ ]:
from PIL import Image
from tqdm import tqdm

corrupt_files = []
total_checked = 0

for class_folder in tqdm(class_folders, desc="Checking classes"):
    for img_path in class_folder.iterdir():
        total_checked += 1
        try:
            with Image.open(img_path) as img:
                img.verify()
        except Exception as e:
            corrupt_files.append((str(img_path), str(e)))

print(f"\nChecked {total_checked} images")
print(f"Corrupt/unreadable: {len(corrupt_files)}")
for path, err in corrupt_files[:20]:
    print(" ", path, "->", err)

### Result

If class folders = 200 with none missing, and corrupt/unreadable = 0 (or a
short, specific list), step 3.1 is done. Any corrupt files found here get
excluded before training later. Next: step 3.2, class distribution.